# 05 — Quench fraction vs angle, as a function of host halo mass

Same satellite cuts as notebook 04 (`log10 M*,sat = 6.0, 6.5, ..., 9.0`); quench fraction vs
azimuthal angle folded to **[0, 90]** (0 = major axis). We look at the host halo-mass dependence
two complementary ways:

* **Cumulative sweep** — lower bound fixed at `log10 M_200c = 12.0`, upper bound grows in 0.25 dex
  (`[12, 12.25], [12, 12.5], ... [12, 14.0]`); each step adds more massive hosts.
* **Disjoint bins** — non-overlapping host-mass bins (default `12-13`, `13-14`, `14+`, set by
  `HOST_LOGM_BIN_EDGES`), so each bin isolates a halo-mass regime.

For each we (i) plot f_q vs angle for the seven satellite cuts, with the fitted sinusoid
`a + b cos(2 theta)` and its uncertainty band overlaid, and (ii) summarize the amplitude `b` (with
its error) vs halo mass, flagging where it is significant. `FIT_METHOD` chooses least-squares (fast,
bootstrap error) or MCMC (emcee, like notebook 03).

**Runs only on Binder** (raw TNG via `tng_utils`); all physics shared through `tng_utils.py`.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import emcee                       # used when FIT_METHOD = 'mcmc'

sys.path.insert(0, os.getcwd())
import tng_utils as T

## Configuration

In [ ]:
SIM      = 'TNG100'                                      # 'TNG100' or 'TNG50'
REDSHIFT = 'z0'                                          # 'z0' (snap 99) or 'z0p05' (snap 98)
TNG_ROOT = T.sim_root(SIM)
SNAP     = T.snap_for(REDSHIFT)

SAT_LOGM_CUTS = np.array([6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0])   # log10 M*,sat lower limits

# cumulative sweep: lower bound fixed, upper bound grows
HOST_LOGM_LOWER  = 12.0
HOST_LOGM_UPPERS = np.arange(12.25, 14.0 + 1e-6, 0.25)  # 12.25, 12.5, ... 14.0

# disjoint host-mass bins (CHANGE ME): edges -> bins 12-13, 13-14, 14+
HOST_LOGM_BIN_EDGES = [12.0, 13.0, 14.0, np.inf]

BINW = 5.0
NBINS_90 = int(90 / BINW)

# fitting
FIT_METHOD = 'lsq'     # 'lsq' (bootstrap least squares) or 'mcmc' (emcee, like notebook 03)
SIG_NSIGMA = 2.0       # |b|/sigma_b threshold to call the amplitude significant
SHOW_FIT   = True      # overlay the fitted sinusoid + uncertainty band on the f_q-vs-angle panels
N_BOOT = 1000          # lsq bootstrap resamples
MCMC_WALKERS, MCMC_STEPS, MCMC_BURN = 16, 3000, 600     # emcee settings (raise for final runs)

assert os.path.isdir(os.path.join(TNG_ROOT, 'output')), f'set TNG_ROOT for {SIM}: {TNG_ROOT} not found'
print(f'SIM = {SIM} | REDSHIFT = {REDSHIFT} (snap {SNAP}) | FIT_METHOD = {FIT_METHOD}')

## Build the catalog once over the full host range

Computed a single time over `12.0 < log10 M_200c < 16.0`; every cumulative range and disjoint bin
below is a sub-selection on `host_m200_phys`, so the simulation is read only once.

In [ ]:
df_all = T.compute_satellite_catalog(TNG_ROOT, HOST_LOGM_LOWER, 16.0, sim=SIM, snap=SNAP)
print(f'{SIM} {REDSHIFT} hosts (12.0 < log10 M200c < 16.0): {df_all.host_id.nunique()}')
print(f'satellites (any mass): {len(df_all)} | host_m200_phys range: '
      f'{df_all.host_m200_phys.min():.2f} .. {df_all.host_m200_phys.max():.2f}')

## Fitting helpers

`fit_amplitude` returns the binned f_q with errors, the amplitude `b` and its uncertainty, and the
model curve (median + 16-84 band) for overlay. Least squares gets its band/error from bootstrap
resampling the satellites; MCMC from the emcee posterior of `a + b cos(2 theta)` + jitter.

In [ ]:
TGRID = np.linspace(0, 90, 200)
COS2G = np.cos(2 * np.radians(TGRID))

def _lsq_ab(fq, cos2c):
    ok = np.isfinite(fq)
    if ok.sum() < 3:
        return np.nan, np.nan
    a, b = np.linalg.lstsq(np.vstack([np.ones(ok.sum()), cos2c[ok]]).T, fq[ok], rcond=None)[0]
    return a, b

def fit_amplitude(angle, quenched, method=None, n_boot=None, seed=0):
    '''Returns (centers, fq, err, b, b_err, model) where model = (ymed, ylo, yhi) on TGRID or None.'''
    method = method or FIT_METHOD
    n_boot = n_boot or N_BOOT
    centers, fq, err, cnt = T.fq_vs_angle(angle, quenched, NBINS_90, 90)
    cos2c = np.cos(2 * np.radians(centers))
    if np.isfinite(fq).sum() < 3:
        return centers, fq, err, np.nan, np.nan, None

    if method == 'lsq':
        a0, b0 = _lsq_ab(fq, cos2c)
        edges = np.linspace(0, 90, NBINS_90 + 1)
        bi = np.digitize(angle, edges) - 1
        keep = (bi >= 0) & (bi < NBINS_90)
        bi, q = bi[keep], np.asarray(quenched, float)[keep]
        n = len(bi)
        rng = np.random.default_rng(seed)
        bs, curves = np.empty(n_boot), []
        for i in range(n_boot):
            s = rng.integers(0, n, n)
            c2 = np.bincount(bi[s], minlength=NBINS_90).astype(float)
            sm = np.bincount(bi[s], weights=q[s], minlength=NBINS_90)
            with np.errstate(invalid='ignore', divide='ignore'):
                fqi = np.where(c2 > 0, sm / c2, np.nan)
            ai, bi_ = _lsq_ab(fqi, cos2c)
            bs[i] = bi_
            if np.isfinite(ai):
                curves.append(ai + bi_ * COS2G)
        curves = np.array(curves)
        model = (a0 + b0 * COS2G, np.percentile(curves, 16, 0), np.percentile(curves, 84, 0))
        return centers, fq, err, b0, np.nanstd(bs), model

    # method == 'mcmc'
    ok = np.isfinite(fq) & np.isfinite(err) & (err > 0)
    x, y, s = centers[ok], fq[ok], err[ok]
    def log_prob(th):
        a, b, f = th
        if not (0 < a < 1 and -1 < b < 1 and -10 < f < 2):
            return -np.inf
        var = s ** 2 + np.exp(f) ** 2
        return -0.5 * np.sum((y - (a + b * np.cos(2 * np.radians(x)))) ** 2 / var + np.log(2 * np.pi * var))
    rng = np.random.default_rng(seed)
    p0 = np.array([np.clip(np.nanmean(y), .01, .99), 0., -3.]) + 1e-3 * rng.standard_normal((MCMC_WALKERS, 3))
    sampler = emcee.EnsembleSampler(MCMC_WALKERS, 3, log_prob)
    sampler.run_mcmc(p0, MCMC_STEPS, progress=False)
    chain = sampler.get_chain(discard=MCMC_BURN, flat=True)
    idx = rng.integers(0, len(chain), 2000)
    curves = chain[idx, 0][:, None] + chain[idx, 1][:, None] * COS2G[None, :]
    model = (np.percentile(curves, 50, 0), np.percentile(curves, 16, 0), np.percentile(curves, 84, 0))
    return centers, fq, err, chain[:, 1].mean(), chain[:, 1].std(), model

In [ ]:
colors = plt.cm.coolwarm(np.linspace(0, 1, len(SAT_LOGM_CUTS)))

def panel_grid(ranges, suptitle):
    '''ranges = list of (lo, hi, label). One panel each; 7 sat-cut curves + (optional) fit+band.
    Returns b_arr, be_arr of shape (n_cut, n_range).'''
    nr = len(ranges)
    ncol = min(4, nr); nrow = int(np.ceil(nr / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.6 * nrow), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).ravel()
    b_arr  = np.full((len(SAT_LOGM_CUTS), nr), np.nan)
    be_arr = np.full_like(b_arr, np.nan)
    for ir, (lo, hi, lab) in enumerate(ranges):
        ax = axes[ir]
        hsel = df_all[(df_all.host_m200_phys >= lo) & (df_all.host_m200_phys < hi)]
        for ic, (logcut, col) in enumerate(zip(SAT_LOGM_CUTS, colors)):
            sub = hsel[hsel.mstar_phys > logcut]
            c, fq, err, b, be, model = fit_amplitude(sub.alpha, sub.quenched)
            b_arr[ic, ir], be_arr[ic, ir] = b, be
            ax.errorbar(c, fq, yerr=err, color=col, marker='o', ms=3, lw=1.1, capsize=1.5)
            if SHOW_FIT and model is not None:
                ymed, ylo, yhi = model
                ax.plot(TGRID, ymed, color=col, lw=1.0)
                ax.fill_between(TGRID, ylo, yhi, color=col, alpha=0.12, lw=0)
        ax.set_xlim(0, 90); ax.set_ylim(0, 1)
        ax.set_title(f'{lab}  (N$_{{host}}$={hsel.host_id.nunique()})', fontsize=10)
        ax.tick_params(which='both', direction='in', top=True, right=True)
    for ax in axes[nr:]:
        ax.axis('off')
    for ax in axes[::ncol]:
        ax.set_ylabel(r'$f_q$')
    for ax in axes[-ncol:]:
        ax.set_xlabel(r'$\theta$ from major axis [deg]')
    handles = [plt.Line2D([], [], color=c, marker='o', label=f'{lc:.1f}')
               for lc, c in zip(SAT_LOGM_CUTS, colors)]
    fig.legend(handles=handles, title=r'$\log_{10} M_{*,\rm sat}$', loc='center right',
               fontsize=9, fancybox=False)
    fig.suptitle(suptitle, y=1.0)
    plt.tight_layout(rect=[0, 0, 0.9, 1]); plt.show()
    return b_arr, be_arr

## (A) Cumulative sweep: f_q vs angle

In [ ]:
ranges_cum = [(HOST_LOGM_LOWER, u, rf'$12.0\!-\!{u:.2f}$') for u in HOST_LOGM_UPPERS]
b_cum, be_cum = panel_grid(ranges_cum,
    f'{SIM} ({REDSHIFT}) cumulative host range — f_q vs angle ({FIT_METHOD.upper()} fit + band)')

## (B) Disjoint host-mass bins: f_q vs angle

Change `HOST_LOGM_BIN_EDGES` in the config cell to use different bins.

In [ ]:
def bins_from_edges(edges):
    out = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        lab = rf'${lo:.1f}\!-\!{hi:.1f}$' if np.isfinite(hi) else rf'$>{lo:.1f}$'
        out.append((lo, hi, lab))
    return out

ranges_bin = bins_from_edges(HOST_LOGM_BIN_EDGES)
b_bin, be_bin = panel_grid(ranges_bin,
    f'{SIM} ({REDSHIFT}) disjoint host-mass bins — f_q vs angle ({FIT_METHOD.upper()} fit + band)')

## (C) Amplitude `b` vs host mass

Amplitude of the `a + b cos(2 theta)` fit (with its uncertainty) for the cumulative sweep (left)
and the disjoint bins (right). **Filled** points are significant (`|b|/sigma_b >= SIG_NSIGMA`),
open are consistent with zero. `b > 0` means more quenching toward the major axis.

In [ ]:
def sig_scatter(ax, x, b, be, col):
    sig = np.abs(b) >= SIG_NSIGMA * be
    ax.errorbar(x, b, yerr=be, color=col, lw=1.4, capsize=2, zorder=2)
    ax.scatter(np.asarray(x)[sig],  np.asarray(b)[sig],  color=col, edgecolor='k', s=50, zorder=3)
    ax.scatter(np.asarray(x)[~sig], np.asarray(b)[~sig], facecolor='white', edgecolor=col, s=42, zorder=3)

fig, (axc, axb) = plt.subplots(1, 2, figsize=(14, 6))
for ic, (logcut, col) in enumerate(zip(SAT_LOGM_CUTS, colors)):
    sig_scatter(axc, HOST_LOGM_UPPERS, b_cum[ic], be_cum[ic], col)
    sig_scatter(axb, np.arange(len(ranges_bin)), b_bin[ic], be_bin[ic], col)
for ax in (axc, axb):
    ax.axhline(0, ls='--', color='grey')
    ax.set_ylabel(r'amplitude $b$  ($f_q = a + b\cos 2\theta$)')
    ax.tick_params(which='both', direction='in', top=True, right=True)
axc.set_xlabel(r'cumulative upper bound  $\log_{10} M_{200c}^{\max}$')
axc.set_title('cumulative sweep')
axb.set_xticks(np.arange(len(ranges_bin)))
axb.set_xticklabels([lab for _, _, lab in ranges_bin])
axb.set_xlabel('host-mass bin'); axb.set_title('disjoint bins')
handles = [plt.Line2D([], [], color=c, marker='o', label=f'{lc:.1f}')
           for lc, c in zip(SAT_LOGM_CUTS, colors)]
fig.legend(handles=handles, title=r'$\log_{10} M_{*,\rm sat}$', loc='center right', fontsize=9, fancybox=False)
fig.suptitle(f'{SIM} ({REDSHIFT}) anisotropy amplitude vs halo mass '
             f'({FIT_METHOD.upper()}; filled = |b|/sigma_b >= {SIG_NSIGMA:.0f})')
plt.tight_layout(rect=[0, 0, 0.9, 1]); plt.show()

# significance tables
def print_grid(name, cols_label, cols, b, be):
    print(f'\n|b|/sigma_b  {name}  (rows = sat cut logM)')
    print('          ' + '  '.join(f'{c:>7}' for c in cols_label))
    for ic, lc in enumerate(SAT_LOGM_CUTS):
        print(f'  logM>{lc:3.1f}: ' + '  '.join(f'{abs(b[ic, j])/be[ic, j]:7.1f}' for j in range(len(cols))))

print_grid('cumulative', [f'{u:.2f}' for u in HOST_LOGM_UPPERS], HOST_LOGM_UPPERS, b_cum, be_cum)
print_grid('disjoint',   [lab.replace('$','').replace('\\!','') for _,_,lab in ranges_bin], ranges_bin, b_bin, be_bin)